# Quy Trình Các Bước Feature Engineering - So Sánh 2 Phương Phương Điền Khuyết & 3 Kịch Bản Biến

Notebook này thực hiện tiền xử lý dữ liệu từng bước trực quan, so sánh 2 phương pháp điền khuyết (Mean vs Median) cho các giá trị khuyết thiếu (NaN), trong khi các đặc trưng hành vi và mã hóa được tính toán bình thường bằng Mean.

Quy trình từng bước chi tiết được hiển thị dưới đây (sử dụng cấu hình **Median Imputation** chính thức). Ở cuối notebook, chúng ta sẽ tự động hóa để xuất thêm các kịch bản đối chứng sử dụng **Mean Imputation**.

## Bước 1. Chia Tách Dữ Liệu (Train/Test Split)

Chia tập dữ liệu thành Train (80%) và Test (20%) trước khi tiền xử lý để tránh rò rỉ dữ liệu.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import shutil

# Load dữ liệu thô
df = pd.read_csv('../data/raw/retail_store_sales.csv')
print('=== KÍCH THƯỚC DỮ LIỆU THÔ ===')
print(f'Số dòng: {df.shape[0]:,} | Số cột: {df.shape[1]}')

# Chia Train/Test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.copy()
test_df = test_df.copy()

print(f'Tập huấn luyện (Train): {train_df.shape}')
print(f'Tập kiểm thử (Test): {test_df.shape}')

=== KÍCH THƯỚC DỮ LIỆU THÔ ===
Số dòng: 12,575 | Số cột: 11
Tập huấn luyện (Train): (10060, 11)
Tập kiểm thử (Test): (2515, 11)


## Bước 2. Xử Lý Giá Trị Thiếu (Handling Missing Values)

### Bước 2.1: Phục hồi đơn giá Price Per Unit và điền khuyết bằng Median

In [2]:
# 1. Khôi phục Price Per Unit bằng Total Spent / Quantity trước (Logic chính xác tuyệt đối)
def impute_price(row):
    if pd.isnull(row['Price Per Unit']) and not pd.isnull(row['Quantity']) and not pd.isnull(row['Total Spent']):
        return row['Total Spent'] / row['Quantity']
    return row['Price Per Unit']

train_df['Price Per Unit'] = train_df.apply(impute_price, axis=1)
test_df['Price Per Unit'] = test_df.apply(impute_price, axis=1)

# 2. Điền khuyết các giá trị Price Per Unit và Total Spent còn lại bằng Median tập Train
price_median = train_df['Price Per Unit'].median()
spent_median = train_df['Total Spent'].median()
print(f'Median Price Per Unit: {price_median:.4f}')
print(f'Median Total Spent: {spent_median:.4f}')

train_df['Price Per Unit'] = train_df['Price Per Unit'].fillna(price_median)
test_df['Price Per Unit'] = test_df['Price Per Unit'].fillna(price_median)
train_df['Total Spent'] = train_df['Total Spent'].fillna(spent_median)
test_df['Total Spent'] = test_df['Total Spent'].fillna(spent_median)

# Hiển thị dữ liệu sau khi xử lý Price và Spent
display(train_df[['Price Per Unit', 'Quantity', 'Total Spent']].head(5))

Median Price Per Unit: 23.0000
Median Total Spent: 108.5000


,Price Per Unit,Quantity,Total Spent
7919,29.0,9.0,261.0
10771,23.0,5.0,115.0
11832,33.5,5.0,167.5
10512,8.0,4.0,32.0
7807,30.5,8.0,244.0


### Bước 2.2: Phục hồi tên mặt hàng Item bằng bản đồ liên kết

In [12]:
# Tạo bản đồ liên kết (Category, Price) -> Item từ tập Train
lookup_df = train_df.dropna(subset=['Item', 'Price Per Unit'])
item_map = lookup_df.groupby(['Category', 'Price Per Unit'])['Item'].first().to_dict()

def restore_item(row):
    if pd.isnull(row['Item']) and not pd.isnull(row['Price Per Unit']):
        key = (row['Category'], row['Price Per Unit'])
        return item_map.get(key, row['Item'])
    return row['Item']

train_df['Item'] = train_df.apply(restore_item, axis=1)
test_df['Item'] = test_df.apply(restore_item, axis=1)

# Điền khuyết các tên Item còn lại bằng Mode tập Train
item_mode = train_df['Item'].mode()[0]
train_df['Item'] = train_df['Item'].fillna(item_mode)
test_df['Item'] = test_df['Item'].fillna(item_mode)

print(f'Mode Item dùng điền khuyết: "{item_mode}"')
display(train_df[['Category', 'Price Per Unit', 'Item']].head(15))

Mode Item dùng điền khuyết: "Item_2_BEV"


,Category,Price Per Unit,Item
0,Milk Products,29.0,Item_17_MILK
1,Milk Products,23.0,Item_13_MILK
2,Electric household essentials,33.5,Item_20_EHE
3,Food,8.0,Item_3_FOOD
4,Milk Products,30.5,Item_18_MILK
5,Butchers,33.5,Item_20_BUT
6,Furniture,26.0,Item_15_FUR
7,Food,21.5,Item_12_FOOD
8,Computers and electric accessories,21.5,Item_12_CEA
9,Food,33.5,Item_20_FOOD


### Bước 2.3: Loại bỏ khuyết Quantity và điền khuyết Discount Applied

Loại bỏ các dòng khuyết biến mục tiêu Quantity và chuyển Discount Applied sang dạng số nhị phân.

In [4]:
train_df = train_df.dropna(subset=['Quantity'])
test_df = test_df.dropna(subset=['Quantity'])

train_df['Discount Applied'] = train_df['Discount Applied'].fillna(False).astype(int)
test_df['Discount Applied'] = test_df['Discount Applied'].fillna(False).astype(int)

print('=== KIỂM TRA SỐ LƯỢNG Ô KHUYẾT CỦA TẬP TRAIN SAU BƯỚC LÀM SẠCH ===')
print(train_df.isnull().sum())

=== KIỂM TRA SỐ LƯỢNG Ô KHUYẾT CỦA TẬP TRAIN SAU BƯỚC LÀM SẠCH ===
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


## Bước 3. Tạo Đặc Trưng Mới (Feature Creation)

### Bước 3.1: Tạo đặc trưng thời gian từ Transaction Date

In [5]:
for df_temp in [train_df, test_df]:
    dates = pd.to_datetime(df_temp['Transaction Date'])
    df_temp['Txn_Year'] = dates.dt.year
    df_temp['Txn_Month'] = dates.dt.month
    df_temp['Txn_Day'] = dates.dt.day
    df_temp['Txn_DayOfWeek'] = dates.dt.dayofweek
    df_temp['Txn_IsWeekend'] = (dates.dt.dayofweek >= 5).astype(int)

display(train_df[['Transaction Date', 'Txn_Year', 'Txn_Month', 'Txn_DayOfWeek', 'Txn_IsWeekend']].head(3))

,Transaction Date,Txn_Year,Txn_Month,Txn_DayOfWeek,Txn_IsWeekend
7919,2023-08-02,2023,8,2,0
10771,2022-05-30,2022,5,0,0
11832,2023-06-07,2023,6,2,0


### Bước 3.2: Tạo đặc trưng tích lũy khách hàng bằng Mean

In [6]:
# Tính toán đặc trưng tích lũy khách hàng dựa trên tập Train
cust_stats = train_df.groupby('Customer ID').agg(
    Customer_Txn_Count=('Transaction ID', 'count'),
    Customer_Avg_Quantity=('Quantity', 'mean'),
    Customer_Avg_Spent=('Total Spent', 'mean')
).reset_index()

# Các giá trị điền khuyết cho khách hàng mới sử dụng Median (Kịch bản chính thức)
global_txn_count = cust_stats['Customer_Txn_Count'].median()
global_avg_qty = cust_stats['Customer_Avg_Quantity'].median()
global_avg_spent = cust_stats['Customer_Avg_Spent'].median()
global_qty_val = train_df['Quantity'].median()

train_df = train_df.merge(cust_stats, on='Customer ID', how='left')
test_df = test_df.merge(cust_stats, on='Customer ID', how='left')

for df_temp in [train_df, test_df]:
    df_temp['Customer_Txn_Count'] = df_temp['Customer_Txn_Count'].fillna(global_txn_count)
    df_temp['Customer_Avg_Quantity'] = df_temp['Customer_Avg_Quantity'].fillna(global_avg_qty)
    df_temp['Customer_Avg_Spent'] = df_temp['Customer_Avg_Spent'].fillna(global_avg_spent)

display(train_df[['Customer ID', 'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent']].head(5))

,Customer ID,Customer_Txn_Count,Customer_Avg_Quantity,Customer_Avg_Spent
0,CUST_21,390,5.566667,128.814103
1,CUST_05,421,5.410926,126.952494
2,CUST_13,408,5.531863,123.968137
3,CUST_12,397,5.410579,126.370277
4,CUST_09,367,5.302452,123.051771


### Bước 3.3: Tạo đặc trưng Target Encoding bằng Mean

In [7]:
item_target_enc = train_df.groupby('Item')['Quantity'].mean().to_dict()
cat_target_enc = train_df.groupby('Category')['Quantity'].mean().to_dict()

train_df['Item_Target_Enc'] = train_df['Item'].map(item_target_enc).fillna(global_qty_val)
test_df['Item_Target_Enc'] = test_df['Item'].map(item_target_enc).fillna(global_qty_val)
train_df['Category_Target_Enc'] = train_df['Category'].map(cat_target_enc).fillna(global_qty_val)
test_df['Category_Target_Enc'] = test_df['Category'].map(cat_target_enc).fillna(global_qty_val)

display(train_df[['Category', 'Item', 'Category_Target_Enc', 'Item_Target_Enc']].head(5))

,Category,Item,Category_Target_Enc,Item_Target_Enc
0,Milk Products,Item_17_MILK,5.441374,6.116667
1,Milk Products,Item_13_MILK,5.441374,5.362500
2,Electric household essentials,Item_20_EHE,5.448020,5.350649
3,Food,Item_3_FOOD,5.467111,5.609756
4,Milk Products,Item_18_MILK,5.441374,5.432432


## Bước 4. Mã Hóa Biến Phân Loại (One-Hot Encoding)

In [8]:
train_df = pd.get_dummies(train_df, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

display(train_df.head(3))

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Transaction Date,Discount Applied,Txn_Year,...,Txn_DayOfWeek,Txn_IsWeekend,Customer_Txn_Count,Customer_Avg_Quantity,Customer_Avg_Spent,Item_Target_Enc,Category_Target_Enc,Payment Method_Credit Card,Payment Method_Digital Wallet,Location_Online
0,TXN_7663927,CUST_21,Milk Products,Item_17_MILK,29.0,9.0,261.0,2023-08-02,1,2023,...,2,0,390,5.566667,128.814103,6.116667,5.441374,0,1,1
1,TXN_2807320,CUST_05,Milk Products,Item_13_MILK,23.0,5.0,115.0,2022-05-30,0,2022,...,0,0,421,5.410926,126.952494,5.362500,5.441374,0,0,1
2,TXN_3887687,CUST_13,Electric household essentials,Item_20_EHE,33.5,5.0,167.5,2023-06-07,0,2023,...,2,0,408,5.531863,123.968137,5.350649,5.448020,0,1,1


## Bước 5. Tạo 3 Kịch Bản Đặc Trưng (c1, c2, c3) & Chuẩn Hóa Dữ Liệu

Chúng ta thực hiện chuẩn hóa (scaling) tập trung cho các cột số và hiển thị mẫu dữ liệu sau khi chuẩn hóa thành công.

In [9]:
cols_to_drop_base = ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Transaction Date']
output_dir = '../data/ready_train/'
os.makedirs(output_dir, exist_ok=True)

def save_scenario(tr, te, name):
    X_tr = tr.drop(columns=cols_to_drop_base + ['Quantity'])
    y_tr = tr['Quantity']
    X_te = te.drop(columns=cols_to_drop_base + ['Quantity'])
    y_te = te['Quantity']
    
    train_out = X_tr.copy()
    train_out['Quantity'] = y_tr
    test_out = X_te.copy()
    test_out['Quantity'] = y_te
    
    train_out.to_csv(os.path.join(output_dir, f'train_{name}.csv'), index=False)
    test_out.to_csv(os.path.join(output_dir, f'test_{name}.csv'), index=False)
    print(f'- Da luu kịch bản {name} (Shape Train: {train_out.shape} | Shape Test: {test_out.shape})')

# === KỊCH BẢN c1: Giữ cả 2 biến ===
num_cols_c1 = [
    'Price Per Unit', 'Total Spent', 'Txn_Year', 'Txn_Month', 'Txn_Day', 'Txn_DayOfWeek', 'Txn_IsWeekend',
    'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent', 'Item_Target_Enc', 'Category_Target_Enc'
]
scaler_c1 = StandardScaler()
train_c1 = train_df.copy()
test_c1 = test_df.copy()
train_c1[num_cols_c1] = scaler_c1.fit_transform(train_c1[num_cols_c1])
test_c1[num_cols_c1] = scaler_c1.transform(test_c1[num_cols_c1])
save_scenario(train_c1, test_c1, 'ready') # Kịch bản chính thức median_c1

# Xem các cột và mẫu dữ liệu sau khi chuẩn hóa kịch bản chính thức (c1)
print('\n=== CÁC CỘT VÀ BẢN GHI MẪU SAU KHI CHUẨN HÓA (KỊCH BẢN C1) ===')
display(train_c1.drop(columns=cols_to_drop_base).head(3))

# === KỊCH BẢN c2: Bỏ Total Spent ===
num_cols_c2 = [
    'Price Per Unit', 'Txn_Year', 'Txn_Month', 'Txn_Day', 'Txn_DayOfWeek', 'Txn_IsWeekend',
    'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent', 'Item_Target_Enc', 'Category_Target_Enc'
]
scaler_c2 = StandardScaler()
train_c2 = train_df.copy().drop(columns=['Total Spent'])
test_c2 = test_df.copy().drop(columns=['Total Spent'])
train_c2[num_cols_c2] = scaler_c2.fit_transform(train_c2[num_cols_c2])
test_c2[num_cols_c2] = scaler_c2.transform(test_c2[num_cols_c2])
save_scenario(train_c2, test_c2, 'median_c2')

# === KỊCH BẢN c3: Gộp bằng PCA ===
pca_scaler = StandardScaler()
price_spent_train = pca_scaler.fit_transform(train_df[['Price Per Unit', 'Total Spent']])
price_spent_test = pca_scaler.transform(test_df[['Price Per Unit', 'Total Spent']])
pca = PCA(n_components=1)
train_pca_feat = pca.fit_transform(price_spent_train)
test_pca_feat = pca.transform(price_spent_test)
print(f'Explained Variance Ratio of PC1: {pca.explained_variance_ratio_[0]:.4f}')

train_c3 = train_df.copy().drop(columns=['Price Per Unit', 'Total Spent'])
test_c3 = test_df.copy().drop(columns=['Price Per Unit', 'Total Spent'])
train_c3['Price_Spent_PCA'] = train_pca_feat
test_c3['Price_Spent_PCA'] = test_pca_feat
num_cols_c3 = [
    'Price_Spent_PCA', 'Txn_Year', 'Txn_Month', 'Txn_Day', 'Txn_DayOfWeek', 'Txn_IsWeekend',
    'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent', 'Item_Target_Enc', 'Category_Target_Enc'
]
scaler_c3 = StandardScaler()
train_c3[num_cols_c3] = scaler_c3.fit_transform(train_c3[num_cols_c3])
test_c3[num_cols_c3] = scaler_c3.transform(test_c3[num_cols_c3])
save_scenario(train_c3, test_c3, 'median_c3')

- Da luu kịch bản ready (Shape Train: (9577, 17) | Shape Test: (2394, 17))

=== CÁC CỘT VÀ BẢN GHI MẪU SAU KHI CHUẨN HÓA (KỊCH BẢN C1) ===


,Price Per Unit,Quantity,Total Spent,Discount Applied,Txn_Year,Txn_Month,Txn_Day,Txn_DayOfWeek,Txn_IsWeekend,Customer_Txn_Count,Customer_Avg_Quantity,Customer_Avg_Spent,Item_Target_Enc,Category_Target_Enc,Payment Method_Credit Card,Payment Method_Digital Wallet,Location_Online
0,0.529042,9.0,1.396370,1,-0.052545,0.470134,-1.543471,-0.505974,-0.634999,0.310927,0.425905,-0.069582,1.595403,-1.024014,0,1,1
1,-0.030359,5.0,-0.149466,0,-1.222832,-0.389937,1.624006,-1.504265,-0.634999,1.924431,-1.034546,-0.497754,-0.425362,-1.024014,0,0,1
2,0.948592,5.0,0.406400,0,-0.052545,-0.103247,-0.977850,-0.505974,-0.634999,1.247801,0.099532,-1.184158,-0.457116,-0.938808,0,1,1


- Da luu kịch bản median_c2 (Shape Train: (9577, 16) | Shape Test: (2394, 16))
Explained Variance Ratio of PC1: 0.8154


- Da luu kịch bản median_c3 (Shape Train: (9577, 16) | Shape Test: (2394, 16))


## Bước 6. Thực Thi Tự Động Hóa Xuất File Cho Bộ Điền Khuyết Mean (Đối Chứng)

### Lý do thực hiện Bước 6:
Để phục vụ bảng so sánh hiệu năng chi tiết giữa hai phương pháp điền khuyết (**Mean vs Median** Imputation) trong phần Modeling, chúng ta cần tạo ra thêm 3 kịch bản tương ứng sử dụng Mean Imputation làm đối chứng.

Thay vì lặp lại thủ công toàn bộ các ô code từ Bước 2 đến Bước 5 cho phương pháp Mean (gây dài dòng và trùng lặp mã nguồn), chúng ta viết một hàm tự động hóa ngắn gọn `export_mean_datasets()` dưới đây để xử lý và lưu trữ nhanh 3 kịch bản đối chứng (`mean_c1`, `mean_c2`, `mean_c3`) trong một lần chạy.

In [10]:
def export_mean_datasets():
    # Chia train/test độc lập để tránh rò rỉ dữ liệu
    tr, te = train_test_split(df, test_size=0.2, random_state=42)
    tr, te = tr.copy(), te.copy()
    
    # 1. Điền khuyết Price Per Unit & Total Spent bằng Mean
    tr['Price Per Unit'] = tr.apply(impute_price, axis=1)
    te['Price Per Unit'] = te.apply(impute_price, axis=1)
    p_mean, s_mean = tr['Price Per Unit'].mean(), tr['Total Spent'].mean()
    tr['Price Per Unit'], te['Price Per Unit'] = tr['Price Per Unit'].fillna(p_mean), te['Price Per Unit'].fillna(p_mean)
    tr['Total Spent'], te['Total Spent'] = tr['Total Spent'].fillna(s_mean), te['Total Spent'].fillna(s_mean)
    
    # 2. Phục hồi Item
    tr['Item'], te['Item'] = tr.apply(restore_item, axis=1).fillna(item_mode), te.apply(restore_item, axis=1).fillna(item_mode)
    tr, te = tr.dropna(subset=['Quantity']), te.dropna(subset=['Quantity'])
    tr['Discount Applied'], te['Discount Applied'] = tr['Discount Applied'].fillna(False).astype(int), te['Discount Applied'].fillna(False).astype(int)
    
    # 3. Trích xuất đặc trưng thời gian
    for temp in [tr, te]:
        dt = pd.to_datetime(temp['Transaction Date'])
        temp['Txn_Year'], temp['Txn_Month'], temp['Txn_Day'], temp['Txn_DayOfWeek'], temp['Txn_IsWeekend'] = dt.dt.year, dt.dt.month, dt.dt.day, dt.dt.dayofweek, (dt.dt.dayofweek >= 5).astype(int)
        
    # 4. Đặc trưng tích lũy khách hàng (Mean)
    c_stats = tr.groupby('Customer ID').agg(Customer_Txn_Count=('Transaction ID', 'count'), Customer_Avg_Quantity=('Quantity', 'mean'), Customer_Avg_Spent=('Total Spent', 'mean')).reset_index()
    tr, te = tr.merge(c_stats, on='Customer ID', how='left'), te.merge(c_stats, on='Customer ID', how='left')
    g_txn, g_qty, g_spent = c_stats['Customer_Txn_Count'].mean(), c_stats['Customer_Avg_Quantity'].mean(), c_stats['Customer_Avg_Spent'].mean()
    for temp in [tr, te]:
        temp['Customer_Txn_Count'] = temp['Customer_Txn_Count'].fillna(g_txn)
        temp['Customer_Avg_Quantity'] = temp['Customer_Avg_Quantity'].fillna(g_qty)
        temp['Customer_Avg_Spent'] = temp['Customer_Avg_Spent'].fillna(g_spent)
        
    # 5. Target Encoding (Mean)
    i_enc = tr.groupby('Item')['Quantity'].mean().to_dict()
    c_enc = tr.groupby('Category')['Quantity'].mean().to_dict()
    g_qty_val_mean = tr['Quantity'].mean()
    tr['Item_Target_Enc'] = tr['Item'].map(i_enc).fillna(g_qty_val_mean)
    te['Item_Target_Enc'] = te['Item'].map(i_enc).fillna(g_qty_val_mean)
    tr['Category_Target_Enc'] = tr['Category'].map(c_enc).fillna(g_qty_val_mean)
    te['Category_Target_Enc'] = te['Category'].map(c_enc).fillna(g_qty_val_mean)
    
    # 6. One-Hot Encoding
    tr = pd.get_dummies(tr, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)
    te = pd.get_dummies(te, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)
    te = te.reindex(columns=tr.columns, fill_value=0)
    
    # 7. Chuẩn hóa & Lưu 3 kịch bản
    tr_c1, te_c1 = tr.copy(), te.copy()
    tr_c1[num_cols_c1] = scaler_c1.fit_transform(tr_c1[num_cols_c1])
    te_c1[num_cols_c1] = scaler_c1.transform(te_c1[num_cols_c1])
    save_scenario(tr_c1, te_c1, 'mean_c1')
    
    tr_c2 = tr.copy().drop(columns=['Total Spent'])
    te_c2 = te.copy().drop(columns=['Total Spent'])
    tr_c2[num_cols_c2] = scaler_c2.fit_transform(tr_c2[num_cols_c2])
    te_c2[num_cols_c2] = scaler_c2.transform(te_c2[num_cols_c2])
    save_scenario(tr_c2, te_c2, 'mean_c2')
    
    p_s_tr = pca_scaler.fit_transform(tr[['Price Per Unit', 'Total Spent']])
    p_s_te = pca_scaler.transform(te[['Price Per Unit', 'Total Spent']])
    tr_pca = pca.fit_transform(p_s_tr)
    te_pca = pca.transform(p_s_te)
    tr_c3, te_c3 = tr.copy().drop(columns=['Price Per Unit', 'Total Spent']), te.copy().drop(columns=['Price Per Unit', 'Total Spent'])
    tr_c3['Price_Spent_PCA'], te_c3['Price_Spent_PCA'] = tr_pca, te_pca
    tr_c3[num_cols_c3] = scaler_c3.fit_transform(tr_c3[num_cols_c3])
    te_c3[num_cols_c3] = scaler_c3.transform(te_c3[num_cols_c3])
    save_scenario(tr_c3, te_c3, 'mean_c3')

export_mean_datasets()
print('\n=== THƯ MỤC CÁC FILE ĐẦU RA READY_TRAIN ===')
print(os.listdir(output_dir))

- Da luu kịch bản mean_c1 (Shape Train: (9577, 17) | Shape Test: (2394, 17))
- Da luu kịch bản mean_c2 (Shape Train: (9577, 16) | Shape Test: (2394, 16))


- Da luu kịch bản mean_c3 (Shape Train: (9577, 16) | Shape Test: (2394, 16))

=== THƯ MỤC CÁC FILE ĐẦU RA READY_TRAIN ===
['test_mean_c1.csv', 'test_mean_c2.csv', 'test_mean_c3.csv', 'test_median_c1.csv', 'test_median_c2.csv', 'test_median_c3.csv', 'test_ready.csv', 'test_ready_c1.csv', 'test_ready_c2.csv', 'test_ready_c3.csv', 'train_mean_c1.csv', 'train_mean_c2.csv', 'train_mean_c3.csv', 'train_median_c1.csv', 'train_median_c2.csv', 'train_median_c3.csv', 'train_ready.csv', 'train_ready_c1.csv', 'train_ready_c2.csv', 'train_ready_c3.csv']
